In [1]:
import xarray as xr
import numpy as np
import glob
import os
import time
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ============================================================
# Settings
# ============================================================

EXPERIMENTS = {
    "esm-piControl": {
        "in_dir": "/nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest",
        "areacello_file": "/nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Ofx/areacello/gn/latest/areacello_Ofx_NorESM2-LM_esm-piControl_r1i1p1f1_gn.nc",
        "out_dir": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-piControl/latest/lowO2_volume_4p6mgL",
        "exp_id": "esm-piControl",
        "grid": "gr",
    },

    "esm-up2p0-swl2p0": {
        "in_dir": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009",
        "areacello_file": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010/areacello_Ofx_NorESM2-LM_esm-up2p0_r1i1p1f1_gn.nc",
        "out_dir": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl2p0/v20251009/lowO2_volume_4p6mgL",
        "exp_id": "esm-up2p0-swl2p0",
        "grid": "gr",
    },

    "esm-up2p0-swl4p0": {
        "in_dir": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl4p0/v20251010",
        "areacello_file": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010/areacello_Ofx_NorESM2-LM_esm-up2p0_r1i1p1f1_gn.nc",
        "out_dir": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl4p0/v20251010/lowO2_volume_4p6mgL",
        "exp_id": "esm-up2p0-swl4p0",
        "grid": "gr",
    },
}

THRESHOLD_MG_L = 4.6
O2_MOLAR_MASS_G_MOL = 31.998
THRESHOLD_MOL_M3 = THRESHOLD_MG_L / O2_MOLAR_MASS_G_MOL


# ============================================================
# Helper functions
# ============================================================

def get_lev_bounds(ds):
    if "lev_bnds" in ds:
        lev_bnds = ds["lev_bnds"]
    elif "lev_bounds" in ds:
        lev_bnds = ds["lev_bounds"]
    else:
        raise KeyError("No vertical bounds variable found: expected lev_bnds or lev_bounds")

    bnd_dim = [d for d in lev_bnds.dims if d != "lev"][0]

    z_top = lev_bnds.isel({bnd_dim: 0})
    z_bot = lev_bnds.isel({bnd_dim: 1})

    return z_top, z_bot


def load_area(areacello_file):
    ds_area = xr.open_dataset(areacello_file)
    area = ds_area["areacello"]

    for dim in list(area.dims):
        if dim not in ["j", "i"]:
            area = area.isel({dim: 0})

    area = area.squeeze(drop=True)
    area = area.where(np.isfinite(area))

    return ds_area, area


def check_area_grid(area, o2):
    if ("j" not in area.dims) or ("i" not in area.dims):
        raise ValueError("areacello does not have j/i dimensions.")

    if area.sizes["j"] != o2.sizes["j"] or area.sizes["i"] != o2.sizes["i"]:
        raise ValueError(
            "areacello and o2 horizontal grids do not match: "
            f"area={area.sizes}, o2_j={o2.sizes['j']}, o2_i={o2.sizes['i']}"
        )


def make_output_filename(infile):
    fname = os.path.basename(infile)
    return fname.replace("o2_Omon", "lowO2_volume_4p6mgL")


def process_one_file(infile, out_dir, area, areacello_file):
    start = time.time()

    outfile = os.path.join(out_dir, make_output_filename(infile))

    print("=" * 80)
    print(f"Input : {infile}")
    print(f"Output: {outfile}")

    if os.path.exists(outfile):
        print("Output already exists. Skipping.")
        return

    ds = xr.open_dataset(
        infile,
        use_cftime=True,
        chunks={"time": 12, "lev": 10}
    )

    o2 = ds["o2"]

    check_area_grid(area, o2)

    z_top, z_bot = get_lev_bounds(ds)
    dz = z_bot - z_top
    dz.attrs["units"] = "m"

    # Grid-cell volume for each vertical layer
    volume = (dz * area).fillna(0)
    volume.attrs["units"] = "m3"

    # OMZ volume per horizontal grid cell
    omz_volume = xr.where(
        o2 < THRESHOLD_MOL_M3,
        volume,
        0.0
    ).sum(dim="lev")

    omz_volume.name = "omz_volume"
    omz_volume.attrs["long_name"] = (
        f"Volume of water column with O2 < {THRESHOLD_MG_L} mg L-1"
    )
    omz_volume.attrs["units"] = "m3"
    omz_volume.attrs["threshold_mg_L"] = THRESHOLD_MG_L
    omz_volume.attrs["threshold_mol_m3"] = THRESHOLD_MOL_M3

    coords = {
        "time": ds["time"],
        "j": ds["j"],
        "i": ds["i"],
    }

    for coord_name in ["latitude", "longitude", "lat", "lon"]:
        if coord_name in ds:
            coords[coord_name] = ds[coord_name]

    out = xr.Dataset(
        {"omz_volume": omz_volume},
        coords=coords,
    )

    out.attrs["description"] = (
        "OMZ volume per horizontal grid cell calculated from raw 3-D o2."
    )
    out.attrs["source_file"] = infile
    out.attrs["areacello_file"] = areacello_file
    out.attrs["definition"] = (
        f"Sum of grid-cell volume where o2 < {THRESHOLD_MG_L} mg L-1 "
        f"({THRESHOLD_MOL_M3:.6f} mol m-3)."
    )

    encoding = {
        "omz_volume": {
            "zlib": True,
            "complevel": 4,
            "_FillValue": 1.0e20,
        }
    }

    print("Computing and saving...")
    out.to_netcdf(outfile, encoding=encoding)

    out.close()
    ds.close()

    elapsed = time.time() - start
    print(f"Finished in {elapsed:.2f} sec ({elapsed / 60:.2f} min)")


# ============================================================
# Main loop
# ============================================================

total_start = time.time()

print("Starting OMZ volume production pipeline")
print(f"Threshold: {THRESHOLD_MG_L} mg L-1 = {THRESHOLD_MOL_M3:.6f} mol m-3")

for exp_name, info in EXPERIMENTS.items():

    print("\n" + "#" * 80)
    print(f"Experiment: {exp_name}")
    print("#" * 80)

    in_dir = info["in_dir"]
    out_dir = info["out_dir"]
    exp_id = info["exp_id"]
    grid = info["grid"]
    areacello_file = info["areacello_file"]

    os.makedirs(out_dir, exist_ok=True)

    files = sorted(glob.glob(
        os.path.join(
            in_dir,
            f"o2_Omon_NorESM2-LM_{exp_id}_r1i1p1f1_{grid}_*.nc"
        )
    ))

    print(f"Found {len(files)} O2 files")
    print(f"Output directory: {out_dir}")

    if len(files) == 0:
        print("No O2 files found. Skipping this experiment.")
        continue

    print("First file:", files[0])
    print("Last file :", files[-1])

    if not os.path.exists(areacello_file):
        print(f"WARNING: areacello file not found: {areacello_file}")
        print("Skipping this experiment.")
        continue

    ds_area, area = load_area(areacello_file)

    for n, infile in enumerate(files, start=1):
        print(f"\n[{n}/{len(files)}] Processing {os.path.basename(infile)}")
        process_one_file(
            infile=infile,
            out_dir=out_dir,
            area=area,
            areacello_file=areacello_file,
        )

    ds_area.close()

total_elapsed = time.time() - total_start

print("=" * 80)
print("All files processed.")
print(f"Total elapsed time: {total_elapsed:.2f} sec ({total_elapsed / 60:.2f} min)")

Starting OMZ volume production pipeline
Threshold: 4.6 mg L-1 = 0.143759 mol m-3

################################################################################
Experiment: esm-piControl
################################################################################
Found 25 O2 files
Output directory: /nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-piControl/latest/lowO2_volume_4p6mgL
First file: /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest/o2_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_185101-186012.nc
Last file : /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest/o2_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_209101-210012.nc

[1/25] Processing o2_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_185101-186012.nc
Input : /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest/o2_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_185101-186012.nc
Output: /nird/datalak

In [3]:
import xarray as xr
import numpy as np
import glob
import os
import time
import warnings

warnings.filterwarnings("ignore", category=DeprecationWarning)

# ============================================================
# Settings
# ============================================================

EXPERIMENTS = {
    "esm-piControl": {
        "in_dir": "/nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest",
        "areacello_file": "/nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Ofx/areacello/gn/latest/areacello_Ofx_NorESM2-LM_esm-piControl_r1i1p1f1_gn.nc",
        "out_dir": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-piControl/latest/omz_volume_2mgL",
        "exp_id": "esm-piControl",
        "grid": "gr",
    },

    "esm-up2p0-swl2p0": {
        "in_dir": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl2p0/v20251009",
        "areacello_file": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010/areacello_Ofx_NorESM2-LM_esm-up2p0_r1i1p1f1_gn.nc",
        "out_dir": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl2p0/v20251009/omz_volume_2mgL",
        "exp_id": "esm-up2p0-swl2p0",
        "grid": "gr",
    },

    "esm-up2p0-swl4p0": {
        "in_dir": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0-swl4p0/v20251010",
        "areacello_file": "/nird/datalake/NS2980K/projects/TipESM/cmor/esm-up2p0/v20251010/areacello_Ofx_NorESM2-LM_esm-up2p0_r1i1p1f1_gn.nc",
        "out_dir": "/nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-up2p0-swl4p0/v20251010/omz_volume_2mgL",
        "exp_id": "esm-up2p0-swl4p0",
        "grid": "gr",
    },
}

THRESHOLD_MG_L = 2.0
O2_MOLAR_MASS_G_MOL = 31.998
THRESHOLD_MOL_M3 = THRESHOLD_MG_L / O2_MOLAR_MASS_G_MOL


# ============================================================
# Helper functions
# ============================================================

def get_lev_bounds(ds):
    if "lev_bnds" in ds:
        lev_bnds = ds["lev_bnds"]
    elif "lev_bounds" in ds:
        lev_bnds = ds["lev_bounds"]
    else:
        raise KeyError("No vertical bounds variable found: expected lev_bnds or lev_bounds")

    bnd_dim = [d for d in lev_bnds.dims if d != "lev"][0]

    z_top = lev_bnds.isel({bnd_dim: 0})
    z_bot = lev_bnds.isel({bnd_dim: 1})

    return z_top, z_bot


def load_area(areacello_file):
    ds_area = xr.open_dataset(areacello_file)
    area = ds_area["areacello"]

    for dim in list(area.dims):
        if dim not in ["j", "i"]:
            area = area.isel({dim: 0})

    area = area.squeeze(drop=True)
    area = area.where(np.isfinite(area))

    return ds_area, area


def check_area_grid(area, o2):
    if ("j" not in area.dims) or ("i" not in area.dims):
        raise ValueError("areacello does not have j/i dimensions.")

    if area.sizes["j"] != o2.sizes["j"] or area.sizes["i"] != o2.sizes["i"]:
        raise ValueError(
            "areacello and o2 horizontal grids do not match: "
            f"area={area.sizes}, o2_j={o2.sizes['j']}, o2_i={o2.sizes['i']}"
        )


def make_output_filename(infile):
    fname = os.path.basename(infile)
    return fname.replace("o2_Omon", "omzvol_2mgL_Omon")


def process_one_file(infile, out_dir, area, areacello_file):
    start = time.time()

    outfile = os.path.join(out_dir, make_output_filename(infile))

    print("=" * 80)
    print(f"Input : {infile}")
    print(f"Output: {outfile}")

    if os.path.exists(outfile):
        print("Output already exists. Skipping.")
        return

    ds = xr.open_dataset(
        infile,
        use_cftime=True,
        chunks={"time": 12, "lev": 10}
    )

    o2 = ds["o2"]

    check_area_grid(area, o2)

    z_top, z_bot = get_lev_bounds(ds)
    dz = z_bot - z_top
    dz.attrs["units"] = "m"

    # Grid-cell volume for each vertical layer
    volume = (dz * area).fillna(0)
    volume.attrs["units"] = "m3"

    # OMZ volume per horizontal grid cell
    omz_volume = xr.where(
        o2 < THRESHOLD_MOL_M3,
        volume,
        0.0
    ).sum(dim="lev")

    omz_volume.name = "omz_volume"
    omz_volume.attrs["long_name"] = (
        f"Volume of water column with O2 < {THRESHOLD_MG_L} mg L-1"
    )
    omz_volume.attrs["units"] = "m3"
    omz_volume.attrs["threshold_mg_L"] = THRESHOLD_MG_L
    omz_volume.attrs["threshold_mol_m3"] = THRESHOLD_MOL_M3

    coords = {
        "time": ds["time"],
        "j": ds["j"],
        "i": ds["i"],
    }

    for coord_name in ["latitude", "longitude", "lat", "lon"]:
        if coord_name in ds:
            coords[coord_name] = ds[coord_name]

    out = xr.Dataset(
        {"omz_volume": omz_volume},
        coords=coords,
    )

    out.attrs["description"] = (
        "OMZ volume per horizontal grid cell calculated from raw 3-D o2."
    )
    out.attrs["source_file"] = infile
    out.attrs["areacello_file"] = areacello_file
    out.attrs["definition"] = (
        f"Sum of grid-cell volume where o2 < {THRESHOLD_MG_L} mg L-1 "
        f"({THRESHOLD_MOL_M3:.6f} mol m-3)."
    )

    encoding = {
        "omz_volume": {
            "zlib": True,
            "complevel": 4,
            "_FillValue": 1.0e20,
        }
    }

    print("Computing and saving...")
    out.to_netcdf(outfile, encoding=encoding)

    out.close()
    ds.close()

    elapsed = time.time() - start
    print(f"Finished in {elapsed:.2f} sec ({elapsed / 60:.2f} min)")


# ============================================================
# Main loop
# ============================================================

total_start = time.time()

print("Starting OMZ volume production pipeline")
print(f"Threshold: {THRESHOLD_MG_L} mg L-1 = {THRESHOLD_MOL_M3:.6f} mol m-3")

for exp_name, info in EXPERIMENTS.items():

    print("\n" + "#" * 80)
    print(f"Experiment: {exp_name}")
    print("#" * 80)

    in_dir = info["in_dir"]
    out_dir = info["out_dir"]
    exp_id = info["exp_id"]
    grid = info["grid"]
    areacello_file = info["areacello_file"]

    os.makedirs(out_dir, exist_ok=True)

    files = sorted(glob.glob(
        os.path.join(
            in_dir,
            f"o2_Omon_NorESM2-LM_{exp_id}_r1i1p1f1_{grid}_*.nc"
        )
    ))

    print(f"Found {len(files)} O2 files")
    print(f"Output directory: {out_dir}")

    if len(files) == 0:
        print("No O2 files found. Skipping this experiment.")
        continue

    print("First file:", files[0])
    print("Last file :", files[-1])

    if not os.path.exists(areacello_file):
        print(f"WARNING: areacello file not found: {areacello_file}")
        print("Skipping this experiment.")
        continue

    ds_area, area = load_area(areacello_file)

    for n, infile in enumerate(files, start=1):
        print(f"\n[{n}/{len(files)}] Processing {os.path.basename(infile)}")
        process_one_file(
            infile=infile,
            out_dir=out_dir,
            area=area,
            areacello_file=areacello_file,
        )

    ds_area.close()

total_elapsed = time.time() - total_start

print("=" * 80)
print("All files processed.")
print(f"Total elapsed time: {total_elapsed:.2f} sec ({total_elapsed / 60:.2f} min)")

Starting OMZ volume production pipeline
Threshold: 2.0 mg L-1 = 0.062504 mol m-3

################################################################################
Experiment: esm-piControl
################################################################################
Found 25 O2 files
Output directory: /nird/datalake/NS2980K/users/yongyub/O2_linearlity/TipESM/cmor/esm-piControl/latest/omz_volume_2mgL
First file: /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest/o2_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_185101-186012.nc
Last file : /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest/o2_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_209101-210012.nc

[1/25] Processing o2_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_185101-186012.nc
Input : /nird/datapeak/NS9034K/CMIP6/CMIP/NCC/NorESM2-LM/esm-piControl/r1i1p1f1/Omon/o2/gr/latest/o2_Omon_NorESM2-LM_esm-piControl_r1i1p1f1_gr_185101-186012.nc
Output: /nird/datalake/NS